# Dataset Cleaning

This script prepares the raw IEEE VIS dataset for analysis by:
1. Removing unnecessary columns
2. Filtering out non-archival papers (PaperType = M)
3. Removing rows with missing essential data
4. Exporting a clean CSV

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Load raw data
RAW_PATH = Path("../data/raw/vispubdata_1990_2024.csv")
OUT_PATH = Path("../data/processed/dataset_clean.csv")

df = pd.read_csv(RAW_PATH)
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loaded: 3,877 rows, 20 columns


In [3]:
# Drop unnecessary columns
DROP_COLS = ["Link", "FirstPage", "LastPage", "AuthorNames"]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# Keep only archival papers (exclude Miscellaneous)
df = df[df["PaperType"].str.strip().str.upper() != "M"]

# Remove rows missing essential fields
REQUIRED_COLS = ["Abstract", "AuthorNames-Deduped", "AuthorAffiliation"]
for col in REQUIRED_COLS:
    df[col] = df[col].astype(str)
    df = df[df[col].str.strip().ne("") & df[col].ne("nan")]

# Remove known problematic DOIs
BAD_DOIS = {"10.1109/visual.1997.663901"}
df["DOI"] = df["DOI"].astype(str).str.strip().str.lower()
df = df[~df["DOI"].isin(BAD_DOIS)]

print(f"Clean: {df.shape[0]:,} rows, {df.shape[1]} columns")

Clean: 3,530 rows, 16 columns


In [4]:
# Save clean dataset
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"Saved: {OUT_PATH}")

Saved: ../data/processed/dataset_clean.csv
